In [1]:
import pandas as pd

fuel = pd.read_csv(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/fuel_prices_clean.csv",
    parse_dates=["ngay"]
)
fuel.head()

,ngay,mat_hang_clean,gia_moi
0,2011-09-01,"Dầu DO 0,05S",21100.0
1,2011-09-01,Xăng RON 95,21800.0
2,2011-10-01,"Dầu DO 0,05S",20400.0
3,2012-03-07,Xăng RON 95,23400.0
4,2012-03-07,"Dầu DO 0,05S",21400.0


## Lựa chọn mặt hàng nhiên liệu

Đề tài sử dụng giá Xăng RON 95 và dầu Diesel làm các biến đại diện cho biến động giá nhiên liệu có liên quan trực tiếp đến chi phí giao thông. Các mặt hàng nhiên liệu khác không được sử dụng trong bước xây dựng dữ liệu đầu vào.

In [2]:
fuel_selected = fuel[
    fuel["mat_hang_clean"].str.contains(
        r"Xăng RON 95|Dầu DO",
        regex=True,
        na=False
    )
].copy()
fuel_selected["mat_hang_clean"].value_counts()

mat_hang_clean
Dầu DO 0,05S        288
Xăng RON 95-III     205
Xăng RON 95         113
Dầu DO 0,001S-V      40
Xăng RON 95-II       17
Xăng RON 95-IV       10
Dầu DO 0,005S-IV     10
Name: count, dtype: int64

In [3]:
fuel_selected.groupby("mat_hang_clean").agg(
    tu_ngay=("ngay", "min"),
    den_ngay=("ngay", "max"),
    so_quan_sat=("gia_moi", "count")
).sort_values("tu_ngay")

,tu_ngay,den_ngay,so_quan_sat
mat_hang_clean,,,
"Dầu DO 0,05S",2011-09-01,2024-12-26,288
Xăng RON 95,2011-09-01,2018-12-21,113
Xăng RON 95-II,2016-04-05,2016-12-20,17
"Dầu DO 0,001S-V",2017-12-15,2024-12-26,40
"Dầu DO 0,005S-IV",2018-08-22,2019-01-01,10
Xăng RON 95-III,2018-08-22,2024-12-26,205
Xăng RON 95-IV,2018-08-22,2019-01-01,10


In [4]:
fuel_core = fuel_selected[
    (fuel_selected["mat_hang_clean"] == "Dầu DO 0,05S")
    |
    (
        (fuel_selected["mat_hang_clean"] == "Xăng RON 95")
        & (fuel_selected["ngay"] < "2019-01-01")
    )
    |
    (
        (fuel_selected["mat_hang_clean"] == "Xăng RON 95-III")
        & (fuel_selected["ngay"] >= "2019-01-01")
    )
].copy()
fuel_core["mat_hang_clean"].value_counts()

mat_hang_clean
Dầu DO 0,05S       288
Xăng RON 95-III    196
Xăng RON 95        113
Name: count, dtype: int64

In [5]:
fuel_core["fuel_type"] = fuel_core["mat_hang_clean"].replace({
    "Dầu DO 0,05S": "Diesel",
    "Xăng RON 95": "RON95",
    "Xăng RON 95-III": "RON95"
})
fuel_core["fuel_type"].value_counts()

fuel_type
RON95     309
Diesel    288
Name: count, dtype: int64

## Chuyển giá nhiên liệu về dữ liệu theo ngày

Giá xăng dầu được ghi nhận tại các thời điểm điều chỉnh. Mỗi mức giá được xem là có hiệu lực từ ngày điều chỉnh cho đến trước lần điều chỉnh tiếp theo. Vì vậy, giá được mở rộng theo ngày bằng phương pháp forward-fill trước khi tính giá trung bình theo tháng.

In [6]:
fuel_daily = (
    fuel_core[["ngay", "fuel_type", "gia_moi"]]
    .sort_values(["fuel_type", "ngay"])
    .set_index("ngay")
    .groupby("fuel_type")["gia_moi"]
    .resample("D")
    .ffill()
    .reset_index()
)
fuel_daily.head(15)

,fuel_type,ngay,gia_moi
0,Diesel,2011-09-01,21100.0
1,Diesel,2011-09-02,21100.0
2,Diesel,2011-09-03,21100.0
3,Diesel,2011-09-04,21100.0
4,Diesel,2011-09-05,21100.0
5,Diesel,2011-09-06,21100.0
6,Diesel,2011-09-07,21100.0
7,Diesel,2011-09-08,21100.0
8,Diesel,2011-09-09,21100.0
9,Diesel,2011-09-10,21100.0


In [7]:
fuel_monthly = (
    fuel_daily
    .assign(MonthYear=fuel_daily["ngay"].dt.to_period("M"))
    .groupby(["MonthYear", "fuel_type"])["gia_moi"]
    .mean()
    .round(2)
    .reset_index()
)

fuel_monthly.head(10)

,MonthYear,fuel_type,gia_moi
0,2011-09,Diesel,21100.0
1,2011-09,RON95,21800.0
2,2011-10,Diesel,20400.0
3,2011-10,RON95,21800.0
4,2011-11,Diesel,20400.0
5,2011-11,RON95,21800.0
6,2011-12,Diesel,20400.0
7,2011-12,RON95,21800.0
8,2012-01,Diesel,20400.0
9,2012-01,RON95,21800.0


In [8]:
fuel_monthly_wide = fuel_monthly.pivot(
    index="MonthYear",
    columns="fuel_type",
    values="gia_moi"
).reset_index()

fuel_monthly_wide.columns.name = None

fuel_monthly_wide.head(10)

,MonthYear,Diesel,RON95
0,2011-09,21100.00,21800.00
1,2011-10,20400.00,21800.00
2,2011-11,20400.00,21800.00
3,2011-12,20400.00,21800.00
4,2012-01,20400.00,21800.00
5,2012-02,20400.00,21800.00
6,2012-03,21206.45,23090.32
7,2012-04,21400.00,23400.00
8,2012-05,21432.26,23522.58
9,2012-06,20506.67,22326.67


In [9]:
fuel_monthly_wide.to_csv(
    "../data/interim/fuel_prices_monthly.csv",
    index=False,
    encoding="utf-8-sig"
)